In [2]:
# [Setup]: 

# Ran the following in Anaconda Prompt:
# git clone https://github.com/burke86/deepdisc.git
# cd deepdisc
# conda create -n deepdisc python=3.10 -y
# conda activate deepdisc
# pip install setuptools==67.8.0
# pip install pybind11
# NOTE: scarlet skipped, does not compile on Windows (optional dependency, not needed for detection)

# Ran the following in Anaconda Prompt:
# nvidia-smi

# If Version CUDA is not 12.1, install older versions to be compatible with torch and relevant packages:
# Ran the following in Anaconda Prompt:
# conda activate deepdisc
# pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

# Ran the following in Anaconda Prompt:
# pip install --no-build-isolation git+https://github.com/facebookresearch/detectron2.git
# mkdir taufit
# echo __version__ = "0.1.0" > taufit\version.py
# echo. > requirements.txt
# pip install --no-build-isolation -e .
# pip install --no-deps --no-build-isolation .
# pip install opencv-python
# pip install scikit-image
# pip install ipympl
# pip install ipywidgets
# pip install astropy photutils matplotlib pandas ipykernel astroquery
# python -m ipykernel install --user --name deepdisc --display-name "Python (deepdisc)"

# Now, in VSCodium, click top right for environment -> "Choose another Kernel" -> "deepdisc"

In [3]:
# 02_B_PyTorch_Algorithm.ipynb : Cell 1

%matplotlib widget

import sys
from pathlib import Path
import pickle
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits as astrofits
from astropy.wcs import WCS
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# Paths
BASE_DIR = Path(r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN")

SCRIPTS_DIR = BASE_DIR / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from plate_scan_utils import (
    detect_sources, detect_plate_errors, classify_plate_quality,
    annotate_errors, categorize_plate, fair_plate_features,
    get_plate_catalog_gaia, get_plate_catalog_simbad,
    match_detected_sources_gaia, match_detected_sources_simbad,
    match_against_paradigm, update_name_cache, resolve_names, suggest_name_at,
)

cutout_dir = BASE_DIR / "data" / "R_CrB" / "cutouts"
manifest_path = cutout_dir.parent / "plate_manifest.csv"

scan_cache_dir = BASE_DIR / "data" / "pytorch_scan"
scan_cache_dir.mkdir(parents=True, exist_ok=True)
PLATE_DB_FILE = scan_cache_dir / "plate_db_pytorch.pkl"
PARADIGM_FILE = scan_cache_dir / "paradigm_marks.pkl"
GAIA_CACHE_FILE = scan_cache_dir / "gaia_cache.pkl"
SIMBAD_CACHE_FILE = scan_cache_dir / "simbad_cache.pkl"
NAME_CACHE_FILE = scan_cache_dir / "gaia_name_cache.pkl"

output_dir = BASE_DIR / "data" / "PyTorch"
img_dir    = output_dir / "images"
mask_dir   = output_dir / "masks"
labels_dir = output_dir / "labels"
img_dir.mkdir(parents=True, exist_ok=True)
mask_dir.mkdir(parents=True, exist_ok=True)
labels_dir.mkdir(parents=True, exist_ok=True)

cutouts = sorted(list(cutout_dir.glob("*.fits")) + list(cutout_dir.glob("*.fit")))
print(f"Found {len(cutouts)} cutouts to process")
if len(cutouts) == 0:
    raise RuntimeError("No FITS files found in cutout directory")

PRESCAN_WORKERS = 8
QUALITY_VERSION = 1  # bump to force every cached plate to be re-derived

# Plate limiting-magnitude lookup, same source as 02_A
plate_limit_lookup = {}
if manifest_path.exists():
    manifest_df = pd.read_csv(manifest_path)
    if "filename" in manifest_df.columns:
        for _, row in tqdm(manifest_df.iterrows(), total=len(manifest_df), desc="Reading plate manifest", unit="row"):
            fname = row.get("filename")
            if not (isinstance(fname, str) and fname):
                continue
            a, t = row.get("lim_mag_apass"), row.get("lim_mag_atlas")
            plate_limit_lookup[Path(fname).name] = {
                "lim_mag_apass": float(a) if pd.notna(a) else None,
                "lim_mag_atlas": float(t) if pd.notna(t) else None,
            }
print(f"Loaded plate limits for {len(plate_limit_lookup)} manifest rows.")

def get_plate_limits(fits_path):
    entry = plate_limit_lookup.get(fits_path.name)
    return (entry["lim_mag_apass"], entry["lim_mag_atlas"]) if entry else (None, None)

_apass_vals = [v["lim_mag_apass"] for v in plate_limit_lookup.values() if v.get("lim_mag_apass") is not None]
_atlas_vals = [v["lim_mag_atlas"] for v in plate_limit_lookup.values() if v.get("lim_mag_atlas") is not None]
LIM_MAG_MEDIAN_APASS = float(np.median(_apass_vals)) if _apass_vals else None
LIM_MAG_MEDIAN_ATLAS = float(np.median(_atlas_vals)) if _atlas_vals else None

# Plate database (source detections + quality only -- no naming here yet)
if PLATE_DB_FILE.exists():
    with open(PLATE_DB_FILE, "rb") as fp:
        plate_db = pickle.load(fp)
else:
    plate_db = {}

# Gaia / SIMBAD / name caches -- keyed like 02_A, persisted across runs so
# repeated plate pointings don't re-query the network.
if GAIA_CACHE_FILE.exists():
    with open(GAIA_CACHE_FILE, "rb") as f:
        gaia_cache = pickle.load(f)
else:
    gaia_cache = {}

if SIMBAD_CACHE_FILE.exists():
    with open(SIMBAD_CACHE_FILE, "rb") as f:
        simbad_cache = pickle.load(f)
else:
    simbad_cache = {}

if NAME_CACHE_FILE.exists():
    with open(NAME_CACHE_FILE, "rb") as f:
        gaia_name_cache = pickle.load(f)
else:
    gaia_name_cache = {}

def save_gaia_cache():
    with open(GAIA_CACHE_FILE, "wb") as f:
        pickle.dump(gaia_cache, f)

def save_simbad_cache():
    with open(SIMBAD_CACHE_FILE, "wb") as f:
        pickle.dump(simbad_cache, f)

def save_name_cache():
    with open(NAME_CACHE_FILE, "wb") as f:
        pickle.dump(gaia_name_cache, f)

def _scan_one_plate(f):
    cached = plate_db.get(str(f))
    if cached is not None and cached.get("quality_version") == QUALITY_VERSION:
        return (f, "skip", None)
    try:
        sources, algorithm, data, data_sub, std, x_col, y_col = detect_sources(f)
        n = 0 if sources is None else len(sources)
        errors = detect_plate_errors(data if sources is not None else astrofits.getdata(f).astype(float))
        lim_apass, lim_atlas = get_plate_limits(f)
        quality = classify_plate_quality(
            errors, n, None, lim_apass, lim_atlas, LIM_MAG_MEDIAN_APASS, LIM_MAG_MEDIAN_ATLAS
        )
        entry = {
            "n": n, "quality": quality, "quality_version": QUALITY_VERSION,
            "errors": errors, "sources": sources, "x_col": x_col, "y_col": y_col,
            "human_verdict": cached.get("human_verdict") if cached else None,
        }
        return (f, "new", entry)
    except Exception as e:
        entry = {
            "n": 0, "quality": "defective", "quality_version": QUALITY_VERSION,
            "errors": {}, "sources": None, "x_col": None, "y_col": None,
            "human_verdict": None, "error": str(e),
        }
        return (f, "new", entry)

def prescan(plates=None, max_workers=PRESCAN_WORKERS):
    target_plates = plates if plates is not None else cutouts
    total = max(len(target_plates), 1)
    errors_seen = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(_scan_one_plate, f): f for f in target_plates}
        pbar = tqdm(as_completed(futures), total=total, desc="Prescanning plates", unit="plate")
        for fut in pbar:
            f = futures[fut]
            try:
                f_ret, action, payload = fut.result()
            except Exception as e:
                errors_seen.append(f"{f.name}: {e}")
                continue
            if action == "new":
                plate_db[str(f)] = payload
                if "error" in payload:
                    errors_seen.append(f"{f.name}: {payload['error']}")
            pbar.set_postfix(cached=len(plate_db))

    with open(PLATE_DB_FILE, "wb") as fp:
        pickle.dump(plate_db, fp)

    print(f"Prescan complete: {len(plate_db)} plates cached.")
    if errors_seen:
        print(f"{len(errors_seen)} plate(s) had errors during scanning:")
        for msg in errors_seen[:20]:
            print(f"  [SCAN ERROR] {msg}")
        if len(errors_seen) > 20:
            print(f"  ...and {len(errors_seen) - 20} more.")

prescan()

Found 10803 cutouts to process


Reading plate manifest:   0%|          | 0/15217 [00:00<?, ?row/s]

Loaded plate limits for 10662 manifest rows.


Prescanning plates:   0%|          | 0/10803 [00:00<?, ?plate/s]

Prescan complete: 10803 plates cached.


In [4]:
# 02_B_PyTorch_Algorithm.ipynb : Cell 2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
from tqdm.notebook import tqdm
import random
import json
import copy

BOX_HALF = 8
VAL_SPLIT = 0.15
MAX_EPOCHS = 60          # upper bound -- early stopping will likely stop before this
EARLY_STOP_PATIENCE = 8  # stop if val loss hasn't improved in this many epochs

def crop_to_match(x, ref):
    _, _, h, w = ref.shape
    return x[:, :, :h, :w]

def pad_to_multiple(img, multiple=16):
    _, h, w = img.shape
    pad_h = (multiple - h % multiple) % multiple
    pad_w = (multiple - w % multiple) % multiple
    return F.pad(img, (0, pad_w, 0, pad_h))

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU(),
                                  nn.Conv2d(out_c, out_c, 3, padding=1), nn.ReLU())
        self.enc1, self.enc2, self.enc3 = block(1, 32), block(32, 64), block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.mid = block(128, 256)
        self.up3, self.dec3 = nn.ConvTranspose2d(256, 128, 2, stride=2), block(256, 128)
        self.up2, self.dec2 = nn.ConvTranspose2d(128, 64, 2, stride=2), block(128, 64)
        self.up1, self.dec1 = nn.ConvTranspose2d(64, 32, 2, stride=2), block(64, 32)
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        m = self.mid(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(m), crop_to_match(e3, self.up3(m))], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), crop_to_match(e2, self.up2(d3))], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), crop_to_match(e1, self.up1(d2))], dim=1))
        return torch.sigmoid(self.out(d1))

class StarSegDataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        self.image_paths, self.mask_paths = image_paths, mask_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(str(self.image_paths[idx]), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        img = pad_to_multiple(torch.tensor(img).unsqueeze(0))
        mask = pad_to_multiple(torch.tensor(mask).unsqueeze(0))
        return img, mask

def _points_to_mask(shape, xs, ys, box_half=BOX_HALF):
    h, w = shape
    mask = np.zeros((h, w), dtype=np.uint8)
    for x, y in zip(xs, ys):
        x0, x1 = max(0, int(x - box_half)), min(w, int(x + box_half))
        y0, y1 = max(0, int(y - box_half)), min(h, int(y + box_half))
        mask[y0:y1, x0:x1] = 255
    return mask

def _save_image_mask_pair(fits_path, xs, ys, out_img_dir, out_mask_dir):
    data = astrofits.getdata(fits_path).astype(float)
    data = np.nan_to_num(data, nan=np.nanmedian(data))
    vmin, vmax = np.percentile(data, [1, 99])
    img_norm = np.clip((data - vmin) / (vmax - vmin + 1e-8), 0, 1)
    img_8bit = (img_norm * 255).astype(np.uint8)
    mask = _points_to_mask(img_8bit.shape, xs, ys)

    img_path = out_img_dir / (fits_path.stem + ".png")
    mask_path = out_mask_dir / (fits_path.stem + "_mask.png")
    cv2.imwrite(str(img_path), img_8bit)
    cv2.imwrite(str(mask_path), mask)
    return img_path, mask_path

def _identify_sources(fits_path, xs, ys, known_labels=None):
    """Guesses a name for each (x, y) source on this plate via WCS +
    Gaia/SIMBAD/paradigm crossmatch -- the same approach 02_A uses. This is
    a catalog lookup, NOT something the UNet learns; the segmentation
    network only ever predicts star/not-star. known_labels, if given, is a
    parallel list of user-provided names (from paradigm marking) that take
    priority over the automatic guess wherever non-empty."""
    hdr = astrofits.getheader(fits_path)
    wcs = WCS(hdr)
    xs_arr, ys_arr = np.array(xs, dtype=float), np.array(ys, dtype=float)
    ra, dec = wcs.pixel_to_world_values(xs_arr, ys_arr)
    ra, dec = np.array(ra, dtype=float), np.array(dec, dtype=float)

    data = astrofits.getdata(fits_path)
    try:
        gaia_catalog = get_plate_catalog_gaia(wcs, data.shape, gaia_cache)
        simbad_catalog = get_plate_catalog_simbad(wcs, data.shape, simbad_cache)
        gaia_names = match_detected_sources_gaia(ra, dec, gaia_catalog)
        simbad_names = match_detected_sources_simbad(ra, dec, simbad_catalog)
        update_name_cache(gaia_names, simbad_names, gaia_name_cache)
        resolved = resolve_names(gaia_names, simbad_names, gaia_name_cache)
    except Exception as e:
        print(f"[IDENTIFY ERROR] {fits_path.name}: {e}")
        resolved = ["Unknown"] * len(xs_arr)

    paradigm_names = match_against_paradigm(ra, dec, paradigm_marks)
    final_names = [p if p != "Unknown" else r for p, r in zip(paradigm_names, resolved)]

    if known_labels is not None:
        final_names = [k if k else f for k, f in zip(known_labels, final_names)]

    return [
        {"x": float(x), "y": float(y), "ra": float(r), "dec": float(d), "name": n}
        for x, y, r, d, n in zip(xs_arr, ys_arr, ra, dec, final_names)
    ]

def _save_labels(fits_path, records, out_labels_dir):
    label_path = out_labels_dir / (fits_path.stem + "_labels.json")
    with open(label_path, "w") as f:
        json.dump(records, f, indent=2)
    return label_path

def build_dataset_and_train():
    image_paths, mask_paths = [], []
    n_named_total, n_source_total = 0, 0

    # 1. Ground truth: your human-corrected paradigm plates. Labels the
    #    user typed in take priority; anything left "Unknown" gets one
    #    more automatic crossmatch attempt.
    paradigm_items = [(f_str, marks) for f_str, marks in paradigm_marks.items() if marks]
    for f_str, marks in tqdm(paradigm_items, desc="Building paradigm (ground-truth) plates", unit="plate"):
        fp = Path(f_str)
        xs, ys = [m["x"] for m in marks], [m["y"] for m in marks]
        known = [m.get("label", "") for m in marks]
        ip, mp = _save_image_mask_pair(fp, xs, ys, img_dir, mask_dir)
        image_paths.append(ip); mask_paths.append(mp)

        records = _identify_sources(fp, xs, ys, known_labels=known)
        _save_labels(fp, records, labels_dir)
        n_named_total += sum(1 for r in records if r["name"] != "Unknown")
        n_source_total += len(records)
    print(f"{len(image_paths)} paradigm (ground-truth) plate(s) added.")

    # 2. Pseudo-labels: every other cached plate that passed the quality
    #    bar (algorithm-detected sources used as weak labels), skipping
    #    anything defective/too_many_errors or explicitly rejected, and
    #    skipping plates already added above. Identity is guessed the
    #    same way as for paradigm plates.
    pseudo_items = [
        (f_str, meta) for f_str, meta in plate_db.items()
        if not (f_str in paradigm_marks and paradigm_marks[f_str])
        and meta.get("human_verdict") != "rejected"
        and meta.get("quality") in ("good_match", "fair")
        and meta.get("sources") is not None
    ]
    n_pseudo = 0
    for f_str, meta in tqdm(pseudo_items, desc="Building pseudo-labeled plates", unit="plate"):
        fp = Path(f_str)
        xs = list(meta["sources"][meta["x_col"]])
        ys = list(meta["sources"][meta["y_col"]])
        ip, mp = _save_image_mask_pair(fp, xs, ys, img_dir, mask_dir)
        image_paths.append(ip); mask_paths.append(mp)
        n_pseudo += 1

        records = _identify_sources(fp, xs, ys)
        _save_labels(fp, records, labels_dir)
        n_named_total += sum(1 for r in records if r["name"] != "Unknown")
        n_source_total += len(records)
    print(f"{n_pseudo} pseudo-labeled plate(s) added (quality-filtered algorithm detections).")
    print(f"Identity crossmatch: {n_named_total} of {n_source_total} source(s) across all plates got a name "
          f"(rest saved as 'Unknown'). Per-plate results in {labels_dir}")

    save_gaia_cache(); save_simbad_cache(); save_name_cache()

    if len(image_paths) < 4:
        print("Too few plates to train on -- label more stars on your paradigm plate(s) or approve more 'fair' plates.")
        return None

    combined = list(zip(image_paths, mask_paths))
    random.shuffle(combined)
    n_val = max(1, int(len(combined) * VAL_SPLIT))
    val_pairs, train_pairs = combined[:n_val], combined[n_val:]

    train_ds = StarSegDataset([p[0] for p in train_pairs], [p[1] for p in train_pairs])
    val_ds = StarSegDataset([p[0] for p in val_pairs], [p[1] for p in val_pairs])
    train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=2)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = nn.BCELoss()

    train_loss_history = []
    val_loss_history = []
    best_val_loss = float("inf")
    best_state_dict = None
    epochs_without_improvement = 0
    stopped_early_at = None

    for epoch in range(MAX_EPOCHS):
        model.train()
        train_loss_sum = 0
        for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Train]"):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)
            loss = loss_fn(preds, masks)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            train_loss_sum += loss.item()
        train_loss_avg = train_loss_sum / max(len(train_loader), 1)

        model.eval()
        val_loss_sum = 0
        with torch.no_grad():
            for imgs, masks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Val]"):
                imgs, masks = imgs.to(device), masks.to(device)
                val_loss_sum += loss_fn(model(imgs), masks).item()
        val_loss_avg = val_loss_sum / max(len(val_loader), 1)

        train_loss_history.append(train_loss_avg)
        val_loss_history.append(val_loss_avg)
        print(f"Epoch {epoch+1}: Train Loss={train_loss_avg:.4f} | Val Loss={val_loss_avg:.4f}")

        if val_loss_avg < best_val_loss - 1e-5:
            best_val_loss = val_loss_avg
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOP_PATIENCE:
                stopped_early_at = epoch + 1
                print(f"No val-loss improvement for {EARLY_STOP_PATIENCE} epochs -- stopping early at epoch {stopped_early_at}.")
                break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    model_path = BASE_DIR / "data" / "PyTorch" / "pytorch_trial_1.pth"
    torch.save(model.state_dict(), model_path)
    if stopped_early_at is not None:
        print(f"Training stopped early at epoch {stopped_early_at} (best val loss {best_val_loss:.4f}). Model saved to {model_path}")
    else:
        print(f"Training ran the full {MAX_EPOCHS} epochs (best val loss {best_val_loss:.4f}). Model saved to {model_path}")

    # Loss curve -- both lines are now per-batch AVERAGES, so they're on
    # the same scale and directly comparable, unlike the earlier summed
    # version.
    epochs_range = list(range(1, len(train_loss_history) + 1))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(epochs_range, train_loss_history, marker="o", label="Train Loss")
    ax.plot(epochs_range, val_loss_history, marker="o", label="Val Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("BCE Loss (average per batch)")
    ax.set_title("Training Progress")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    return {"train_loss": train_loss_history, "val_loss": val_loss_history, "model_path": model_path}

In [5]:
# 02_B_PyTorch_Algorithm.ipynb : Cell 3

import ipywidgets as widgets
from IPython.display import display, clear_output
from io import BytesIO
from threading import Thread

REVIEW_CLUSTERS = 20  # hard cap on representative plates shown per batch
PARADIGM_CLICK_PX = 15

# paradigm_marks: { plate_path_str: [ {x, y, ra, dec, label, source}, ... ] }
if PARADIGM_FILE.exists():
    with open(PARADIGM_FILE, "rb") as f:
        _loaded = pickle.load(f)
else:
    _loaded = {}

# Migrate old format (plain (x, y) tuples with no label) to the new dict
# format, so previously-marked plates aren't lost.
paradigm_marks = {}
for f_str, entries in _loaded.items():
    fixed = []
    for e in entries:
        if isinstance(e, dict):
            fixed.append(e)
        else:
            x, y = e
            fixed.append({"x": float(x), "y": float(y), "ra": None, "dec": None, "label": "", "source": "legacy"})
    paradigm_marks[f_str] = fixed

def save_paradigm_marks():
    with open(PARADIGM_FILE, "wb") as f:
        pickle.dump(paradigm_marks, f)

# ---------- Review queue (clustered "fair" plates, capped at 20 PER BATCH) ----------

_fair_cluster_of = {}
_fair_cluster_members = {}
_review_batch_reps = []

def _cluster_fair_plates(n_clusters=REVIEW_CLUSTERS):
    global _fair_cluster_of, _fair_cluster_members
    _fair_cluster_of, _fair_cluster_members = {}, {}

    fair_items = [
        (f_str, meta) for f_str, meta in plate_db.items()
        if meta.get("quality") == "fair" and meta.get("human_verdict") is None
        and meta.get("sources") is not None
    ]
    if not fair_items:
        return []
    if len(fair_items) <= 1:
        f_str = fair_items[0][0]
        _fair_cluster_of[f_str] = 0
        _fair_cluster_members[0] = [f_str]
        return [f_str]

    feats = np.array([
        fair_plate_features(
            meta["errors"], meta["n"], *get_plate_limits(Path(f_str)),
            LIM_MAG_MEDIAN_APASS, LIM_MAG_MEDIAN_ATLAS
        )
        for f_str, meta in fair_items
    ])
    means, stds = feats.mean(axis=0), feats.std(axis=0)
    stds[stds < 1e-9] = 1.0
    feats_z = np.nan_to_num((feats - means) / stds)

    k = max(1, min(n_clusters, len(fair_items)))
    try:
        from scipy.cluster.vq import kmeans2
        _, labels = kmeans2(feats_z, k, minit="++", seed=0)
    except Exception as e:
        print(f"[CLUSTERING] {e} -- falling back to round-robin")
        labels = np.arange(len(fair_items)) % k

    reps = []
    for cid in range(k):
        idx = np.where(labels == cid)[0]
        if len(idx) == 0:
            continue
        members = [fair_items[i][0] for i in idx]
        for m in members:
            _fair_cluster_of[m] = cid
        _fair_cluster_members[cid] = members
        centroid = feats_z[idx].mean(axis=0)
        rep_idx = idx[int(np.argmin(np.linalg.norm(feats_z[idx] - centroid, axis=1)))]
        reps.append(fair_items[rep_idx][0])
    return reps

review_dropdown = widgets.Dropdown(description="Needs review:")
review_status = widgets.Label(value="")
review_output = widgets.Output()
approve_btn = widgets.Button(description="Approve", button_style="success")
reject_btn = widgets.Button(description="Reject", button_style="danger")

def _refresh_review_queue():
    global _review_batch_reps
    if not _review_batch_reps:
        _review_batch_reps = _cluster_fair_plates()

    options = []
    for f_str in _review_batch_reps:
        meta = plate_db.get(f_str)
        if meta is None:
            continue
        cid = _fair_cluster_of.get(f_str)
        n_members = len(_fair_cluster_members.get(cid, [f_str]))
        label = f"{Path(f_str).name} | represents {n_members} plate(s) | {meta['n']} sources"
        options.append((label, Path(f_str)))
    review_dropdown.options = options

    remaining_fair_total = sum(
        1 for meta in plate_db.values()
        if meta.get("quality") == "fair" and meta.get("human_verdict") is None and meta.get("sources") is not None
    )
    review_status.value = (
        f"{len(options)} representative plate(s) in this batch (capped at {REVIEW_CLUSTERS}) -- "
        f"{remaining_fair_total} total 'fair' plate(s) still unreviewed across all batches."
    )

def _open_review_plate(change):
    with review_output:
        clear_output(wait=True)
        fp = review_dropdown.value
        if fp is None:
            print("Nothing to review.")
            return
        meta = plate_db[str(fp)]
        data = astrofits.getdata(fp)
        fig, ax = plt.subplots(figsize=(6, 6))
        ax.imshow(data, origin="lower", cmap="gray")
        if meta.get("sources") is not None:
            ax.scatter(meta["sources"][meta["x_col"]], meta["sources"][meta["y_col"]], s=30, facecolors="none", edgecolors="red")
        annotate_errors(ax, meta["errors"])
        ax.set_title(fp.name)
        buf = BytesIO(); fig.savefig(buf, format="png", bbox_inches="tight"); plt.close(fig); buf.seek(0)
        display(widgets.Image(value=buf.read(), format="png"))

def _approve(_):
    fp = review_dropdown.value
    if fp is None: return
    for m in _fair_cluster_members.get(_fair_cluster_of.get(str(fp)), [str(fp)]):
        plate_db[m]["human_verdict"] = "approved"
    with open(PLATE_DB_FILE, "wb") as f: pickle.dump(plate_db, f)
    if str(fp) in _review_batch_reps:
        _review_batch_reps.remove(str(fp))
    _refresh_review_queue()
    rebuild_paradigm_dropdown()

def _reject(_):
    fp = review_dropdown.value
    if fp is None: return
    for m in _fair_cluster_members.get(_fair_cluster_of.get(str(fp)), [str(fp)]):
        plate_db[m]["human_verdict"] = "rejected"
    with open(PLATE_DB_FILE, "wb") as f: pickle.dump(plate_db, f)
    if str(fp) in _review_batch_reps:
        _review_batch_reps.remove(str(fp))
    _refresh_review_queue()
    rebuild_paradigm_dropdown()

review_dropdown.observe(_open_review_plate, names="value")
approve_btn.on_click(_approve)
reject_btn.on_click(_reject)

display(widgets.VBox([
    widgets.HTML("<b>Review uncertain plates</b> (approve = treat as good, reject = exclude from paradigm picker)"),
    review_dropdown, review_status, widgets.HBox([approve_btn, reject_btn]), review_output,
]))
_refresh_review_queue()

# ---------- Paradigm plate filter + selection ----------

category_filter = widgets.Dropdown(options=["all", "ideal", "good_no_target", "defective_no_target"], value="all", description="Filter:")
search_box = widgets.Text(description="Search:", placeholder="filter by filename")
paradigm_dropdown = widgets.Dropdown(description="Paradigm plate:")
paradigm_dropdown_status = widgets.Label(value="")

def filter_plates(cat_value, query):
    query = query.strip().lower()
    matches = []
    for f_str, meta in plate_db.items():
        if meta.get("human_verdict") == "rejected":
            continue
        cat = categorize_plate(meta)
        if cat_value != "all" and cat != cat_value:
            continue
        f = Path(f_str)
        if query and query not in f.name.lower():
            continue
        label = f"{f.name} | {cat} | {meta['n']} sources"
        matches.append((label, f))
    return matches

def rebuild_paradigm_dropdown(*args):
    matches = filter_plates(category_filter.value, search_box.value)
    paradigm_dropdown.options = matches
    paradigm_dropdown_status.value = f"{len(matches)} matches"

category_filter.observe(rebuild_paradigm_dropdown, names="value")
search_box.observe(rebuild_paradigm_dropdown, names="value")
rebuild_paradigm_dropdown()

# ---------- Paradigm marking + naming (click to add, box prompt to name) ----------

_pstate = {"fig": None, "ax": None, "scatter": None, "fits_path": None, "wcs": None, "entries": []}

paradigm_plot_output = widgets.Output()
paradigm_label_output = widgets.Output()   # the name box prompt appears here
paradigm_table_output = widgets.Output()   # running list of markers + labels
paradigm_msg = widgets.Output()
auto_load_btn = widgets.Button(description="Auto-Load Stars", button_style="info")
clear_btn = widgets.Button(description="Clear Plate", button_style="danger")
train_btn = widgets.Button(description="Build Training Set & Train Model", button_style="success")
train_status = widgets.Label(value="")
train_output = widgets.Output()  # ALL training output (progress bars, print
                                  # statements, and the loss-curve plot from
                                  # build_dataset_and_train()) renders here,
                                  # instead of scattering across cells.

def _entry_xy():
    xs = [e["x"] for e in _pstate["entries"]]
    ys = [e["y"] for e in _pstate["entries"]]
    return np.array(xs), np.array(ys)

def _nearest_entry(x, y, max_px=PARADIGM_CLICK_PX):
    if not _pstate["entries"]:
        return None
    xs, ys = _entry_xy()
    d = np.hypot(xs - x, ys - y)
    j = int(np.argmin(d))
    return j if d[j] <= max_px else None

def _redraw():
    ax = _pstate["ax"]
    if _pstate["scatter"] is not None:
        _pstate["scatter"].remove()
        _pstate["scatter"] = None
    if _pstate["entries"]:
        xs, ys = _entry_xy()
        colors = [
            "limegreen" if (e.get("label") and e["label"] != "Unknown") else "orange"
            for e in _pstate["entries"]
        ]
        _pstate["scatter"] = ax.scatter(xs, ys, s=50, facecolors="none", edgecolors=colors, linewidths=1.5)
    ax.figure.canvas.draw_idle()

def _refresh_paradigm_table():
    with paradigm_table_output:
        clear_output(wait=True)
        if not _pstate["entries"]:
            print("No markers on this plate yet.")
            return
        print(f"{len(_pstate['entries'])} marker(s):")
        for i, e in enumerate(_pstate["entries"]):
            lbl = e["label"] if e["label"] else "(unlabeled)"
            print(f"  [{i}] {lbl:25s} pixel=({e['x']:.1f}, {e['y']:.1f})  source={e['source']}")

def _open_label_editor(entry):
    label_type = widgets.ToggleButtons(options=["Object ID", "GAIA ID"], value="Object ID", description="Type:")
    label_box = widgets.Text(
        value=entry["label"] if entry["label"] and entry["label"] != "Unknown" else "",
        description="Label:",
        placeholder="looking up suggestion... (or just start typing)"
    )
    confirm_btn = widgets.Button(description="Confirm", button_style="success")
    skip_btn = widgets.Button(description="Skip", button_style="")

    def _confirm(_):
        raw = label_box.value.strip()
        if label_type.value == "GAIA ID":
            digits = raw.replace("Gaia", "").strip()
            if digits.isdigit():
                final_label = f"Gaia {digits}"
            else:
                with paradigm_label_output:
                    clear_output(wait=True)
                    print(f"'{raw}' isn't a numeric Gaia source_id -- enter digits only, e.g. 123456789012345.")
                return
        else:
            final_label = raw

        entry["label"] = final_label
        entry["source"] = "manual"
        save_paradigm_marks()
        _redraw()
        _refresh_paradigm_table()
        with paradigm_label_output:
            clear_output(wait=True)
            print(f"Saved: {final_label}")

    def _skip(_):
        with paradigm_label_output:
            clear_output(wait=True)

    confirm_btn.on_click(_confirm)
    skip_btn.on_click(_skip)

    with paradigm_label_output:
        clear_output(wait=True)
        print(f"Pixel ({entry['x']:.1f}, {entry['y']:.1f})  ->  RA={entry['ra']:.5f}, Dec={entry['dec']:.5f}")
        if not entry["label"] or entry["label"] == "Unknown":
            print("Looking up a name suggestion in the background...")
        display(widgets.HBox([label_type, label_box, confirm_btn, skip_btn]))

    def _do_lookup():
        if entry["label"] and entry["label"] != "Unknown":
            return  # already has a real label -- don't overwrite with a suggestion
        try:
            suggestion, src = suggest_name_at(entry["ra"], entry["dec"])
        except Exception as e:
            suggestion, src = "Unknown", f"error: {e}"
        if not label_box.value.strip():
            suggested_mode = "GAIA ID" if suggestion.startswith("Gaia ") else "Object ID"
            label_type.value = suggested_mode
            label_box.value = suggestion.replace("Gaia ", "") if suggested_mode == "GAIA ID" else suggestion
        with paradigm_label_output:
            print(f"Auto-suggested: '{suggestion}' (source: {src})")

    Thread(target=_do_lookup, daemon=True).start()

def _on_click(event):
    if event.inaxes != _pstate["ax"] or event.xdata is None:
        return
    wcs = _pstate["wcs"]

    if event.button == 3:  # right-click = remove
        j = _nearest_entry(event.xdata, event.ydata)
        if j is not None:
            _pstate["entries"].pop(j)
            paradigm_marks[str(_pstate["fits_path"])] = _pstate["entries"]
            save_paradigm_marks()
            _redraw()
            _refresh_paradigm_table()
            with paradigm_msg:
                clear_output(wait=True); print("Removed marker.")
        return

    # left-click: edit existing nearby marker, or add a new one
    j = _nearest_entry(event.xdata, event.ydata)
    if j is not None:
        _open_label_editor(_pstate["entries"][j])
        return

    x, y = float(event.xdata), float(event.ydata)
    ra, dec = wcs.pixel_to_world_values(x, y)
    ra, dec = float(np.array(ra)), float(np.array(dec))
    entry = {"x": x, "y": y, "ra": ra, "dec": dec, "label": "", "source": "manual"}
    _pstate["entries"].append(entry)
    paradigm_marks[str(_pstate["fits_path"])] = _pstate["entries"]
    save_paradigm_marks()
    _redraw()
    _refresh_paradigm_table()
    _open_label_editor(entry)

def _open_paradigm(change):
    fp = paradigm_dropdown.value
    if fp is None:
        return
    data = astrofits.getdata(fp)
    hdr = astrofits.getheader(fp)
    wcs = WCS(hdr)
    entries = paradigm_marks.get(str(fp), [])

    _pstate.update({"fits_path": fp, "wcs": wcs, "entries": entries, "scatter": None})

    with paradigm_plot_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 7))
        ax.imshow(data, origin="lower", cmap="gray")
        ax.set_title(f"{fp.name}  (left-click=add/edit star, right-click=remove)")
        fig.canvas.mpl_connect("button_press_event", _on_click)
        _pstate["fig"], _pstate["ax"] = fig, ax
        _redraw()
        plt.show()

    with paradigm_label_output:
        clear_output(wait=True)
    _refresh_paradigm_table()

def _auto_load(_):
    fp = _pstate["fits_path"]
    if fp is None:
        with paradigm_msg:
            clear_output(wait=True); print("Open a paradigm plate first.")
        return
    meta = plate_db.get(str(fp))
    if meta is None or meta.get("sources") is None:
        with paradigm_msg:
            clear_output(wait=True); print("No cached detections for this plate.")
        return

    wcs = _pstate["wcs"]
    new_xs = np.array(meta["sources"][meta["x_col"]], dtype=float)
    new_ys = np.array(meta["sources"][meta["y_col"]], dtype=float)

    existing_xs, existing_ys = _entry_xy()
    added_x, added_y = [], []
    for nx, ny in zip(new_xs, new_ys):
        if len(existing_xs) > 0:
            d = np.hypot(existing_xs - nx, existing_ys - ny)
            if d.min() <= PARADIGM_CLICK_PX:
                continue
        added_x.append(nx); added_y.append(ny)
        existing_xs = np.append(existing_xs, nx)
        existing_ys = np.append(existing_ys, ny)

    if not added_x:
        with paradigm_msg:
            clear_output(wait=True); print("No new detections to add (all too close to existing markers).")
        return

    added_x, added_y = np.array(added_x), np.array(added_y)
    ra, dec = wcs.pixel_to_world_values(added_x, added_y)
    ra, dec = np.array(ra, dtype=float), np.array(dec, dtype=float)

    with paradigm_msg:
        clear_output(wait=True); print(f"Auto-loaded {len(added_x)} detection(s). Looking up names via Gaia/SIMBAD...")

    try:
        data = astrofits.getdata(fp)
        gaia_catalog = get_plate_catalog_gaia(wcs, data.shape, gaia_cache)
        simbad_catalog = get_plate_catalog_simbad(wcs, data.shape, simbad_cache)
        gaia_names = match_detected_sources_gaia(ra, dec, gaia_catalog)
        simbad_names = match_detected_sources_simbad(ra, dec, simbad_catalog)
        update_name_cache(gaia_names, simbad_names, gaia_name_cache)
        resolved = resolve_names(gaia_names, simbad_names, gaia_name_cache)
        save_gaia_cache(); save_simbad_cache(); save_name_cache()
        n_named = sum(1 for r in resolved if r != "Unknown")
    except Exception as e:
        resolved = ["Unknown"] * len(added_x)
        n_named = 0
        with paradigm_msg:
            print(f"[NAME LOOKUP ERROR] {e}")

    for x, y, r, ra_i, dec_i in zip(added_x, added_y, resolved, ra, dec):
        _pstate["entries"].append({
            "x": float(x), "y": float(y), "ra": float(ra_i), "dec": float(dec_i),
            "label": r, "source": "auto (Gaia/SIMBAD)"
        })

    paradigm_marks[str(fp)] = _pstate["entries"]
    save_paradigm_marks()
    _redraw()
    _refresh_paradigm_table()

    with paradigm_msg:
        clear_output(wait=True)
        print(f"Auto-loaded {len(added_x)} detection(s): {n_named} auto-named, {len(added_x) - n_named} still 'Unknown'.")
        print("Left-click an orange (unnamed) marker to name it manually. Right-click to remove.")

def _clear_plate(_):
    fp = _pstate["fits_path"]
    if fp is None:
        with paradigm_msg:
            clear_output(wait=True); print("No plate is currently open for labeling.")
        return
    n_markers = len(_pstate["entries"])
    _pstate["entries"] = []
    paradigm_marks.pop(str(fp), None)
    save_paradigm_marks()
    _redraw()
    _refresh_paradigm_table()
    with paradigm_msg:
        clear_output(wait=True); print(f"Cleared {n_markers} marker(s) from this plate.")

def _on_train_click(_):
    n_marked = sum(1 for v in paradigm_marks.values() if len(v) > 0)
    if n_marked == 0:
        train_status.value = "Mark at least one star on a paradigm plate first."
        return
    train_status.value = "Building training set and training model... (see output below)"
    with train_output:
        clear_output(wait=True)
        build_dataset_and_train()  # defined in Cell 2 -- run Cell 2 before clicking this
    train_status.value = "Done -- see training log and loss curve below."

auto_load_btn.on_click(_auto_load)
clear_btn.on_click(_clear_plate)
train_btn.on_click(_on_train_click)
paradigm_dropdown.observe(_open_paradigm, names="value")

display(widgets.VBox([
    widgets.HTML(
        "<b>Paradigm plate labeling</b> -- pick a plate, left-click to add a star "
        "(a name box appears -- type a name or wait for the auto-suggestion, then Confirm or Skip), "
        "left-click an existing marker to edit its name, right-click to remove it."
    ),
    category_filter, search_box, paradigm_dropdown, paradigm_dropdown_status,
    widgets.HBox([auto_load_btn, clear_btn]),
    paradigm_plot_output, paradigm_label_output, paradigm_table_output, paradigm_msg,
    train_btn, train_status, train_output,
]))

In [6]:
# After running, the following is generated : 
#   - .png : the plates as normalized 8-bit grayscale images.
#   - _mask.png : a black image with white boxes at every star position (ground truth for the UNet's segmentation training).
#   - .json : a list of dicts, one per star on that plate, each shaped like {"x": 412.3, "y": 187.9, "ra": 237.1442, "dec": 28.1571, "name": "R CrB"}
#   * .pth : the trained PyTorch UNet weights (star/not-star segmentation only, this file has no knowledge of star names, only pixel positions).
#   [Not file, but can be saved] A plot of the losses over time/epochs, to see if the runs are improving the CNN to the model/paradigm plate.

In [10]:
# 02_B_PyTorch_Algorithm.ipynb : Cell 4

import torch
from scipy import ndimage as ndi
import json

PRED_MASK_THRESHOLD = 0.5  # model output at or above this counts as a star
MIN_BLOB_AREA = 4  # ignore predicted specks smaller than this many pixels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_trained_model(model_path=None):
    if model_path is None:
        model_path = BASE_DIR / "data" / "PyTorch" / "pytorch_trial_1.pth"
    model = UNet().to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print(f"Loaded model from {model_path}")
    return model

_inference_model = {"model": None}

def _get_model():
    if _inference_model["model"] is None:
        _inference_model["model"] = load_trained_model()
    return _inference_model["model"]

def _predict_mask(fits_path, model):
    data = astrofits.getdata(fits_path).astype(float)
    data = np.nan_to_num(data, nan=np.nanmedian(data))
    vmin, vmax = np.percentile(data, [1, 99])
    img_norm = np.clip((data - vmin) / (vmax - vmin + 1e-8), 0, 1)
    img_8bit = (img_norm * 255).astype(np.uint8)

    img_t = torch.tensor(img_8bit.astype(np.float32) / 255.0).unsqueeze(0)
    orig_h, orig_w = img_t.shape[1], img_t.shape[2]
    img_t = pad_to_multiple(img_t).unsqueeze(0).to(device)

    with torch.no_grad():
        pred = model(img_t)
    pred = pred.squeeze().cpu().numpy()
    pred = pred[:orig_h, :orig_w]  # crop padding back off
    return img_8bit, pred

def _mask_to_points(pred_mask, threshold=PRED_MASK_THRESHOLD, min_area=MIN_BLOB_AREA):
    binary = (pred_mask >= threshold).astype(np.uint8)
    labeled, n_obj = ndi.label(binary)
    xs, ys = [], []
    for obj_id in range(1, n_obj + 1):
        region = labeled == obj_id
        area = region.sum()
        if area < min_area:
            continue
        coords = np.argwhere(region)
        cy, cx = coords.mean(axis=0)
        xs.append(float(cx)); ys.append(float(cy))
    return xs, ys

def run_inference_on_plate(fits_path, identify=True, save=True):
    # no plotting here, Cell 5's viewer handles display
    model = _get_model()
    img_8bit, pred_mask = _predict_mask(fits_path, model)
    xs, ys = _mask_to_points(pred_mask)

    records = None
    if identify and len(xs) > 0:
        records = _identify_sources(fits_path, xs, ys)
        if save:
            out_path = labels_dir / (fits_path.stem + "_predicted_labels.json")
            with open(out_path, "w") as f:
                json.dump(records, f, indent=2)

    return xs, ys, records

In [11]:
# 02_B_PyTorch_Algorithm.ipynb : Cell 5

import ipywidgets as widgets
from IPython.display import display, clear_output
from io import BytesIO
import json

viewer_mode = widgets.ToggleButtons(
    options=["Algorithm Detections", "Model Predictions"],
    value="Algorithm Detections",
    description="Source:"
)

viewer_error_toggle = widgets.ToggleButtons(
    options=["Show Errors", "Hide Errors"],
    value="Show Errors",
    description="Errors:"
)

viewer_category_filter = widgets.Dropdown(
    options=["all", "ideal", "good_no_target", "defective_no_target"],
    value="all",
    description="Filter:"
)
viewer_search_box = widgets.Text(description="Search:", placeholder="filter by filename")
viewer_dropdown = widgets.Dropdown(description="Plate:")
viewer_dropdown_status = widgets.Label(value="")
viewer_status = widgets.Label(value="Ready")
run_inference_btn = widgets.Button(description="Run Inference on This Plate", button_style="info")
viewer_output = widgets.Output()

def rebuild_viewer_dropdown(*args):
    matches = filter_plates(viewer_category_filter.value, viewer_search_box.value)
    viewer_dropdown.options = matches
    viewer_dropdown_status.value = f"{len(matches)} matches"

viewer_category_filter.observe(rebuild_viewer_dropdown, names="value")
viewer_search_box.observe(rebuild_viewer_dropdown, names="value")

def _load_algorithm_labels(fits_path):
    # saved during build_dataset_and_train, one entry per algorithm detected source, same order
    label_path = labels_dir / (fits_path.stem + "_labels.json")
    if not label_path.exists():
        return None
    with open(label_path, "r") as f:
        return json.load(f)

def _load_predicted_labels(fits_path):
    # saved by run_inference_on_plate, one entry per model predicted source
    label_path = labels_dir / (fits_path.stem + "_predicted_labels.json")
    if not label_path.exists():
        return None
    with open(label_path, "r") as f:
        return json.load(f)

def show_viewer_plate(change=None):
    fits_path = viewer_dropdown.value
    if fits_path is None:
        with viewer_output:
            clear_output(wait=True)
            print("No plate selected.")
        return

    meta = plate_db.get(str(fits_path))
    if meta is None:
        with viewer_output:
            clear_output(wait=True)
            print("This plate hasn't been scanned yet. Run Cell 1's prescan first.")
        return

    viewer_status.value = "Rendering..."
    data = astrofits.getdata(fits_path)

    if viewer_mode.value == "Algorithm Detections":
        if meta.get("sources") is None:
            with viewer_output:
                clear_output(wait=True)
                print("No algorithm detections cached for this plate.")
            viewer_status.value = "Ready"
            return
        xs = list(meta["sources"][meta["x_col"]])
        ys = list(meta["sources"][meta["y_col"]])
        records = _load_algorithm_labels(fits_path)
        if records is not None and len(records) == len(xs):
            names = [r["name"] for r in records]
        else:
            names = ["Unlabeled"] * len(xs)
    else:
        records = _load_predicted_labels(fits_path)
        if records is None:
            with viewer_output:
                clear_output(wait=True)
                print("No model predictions saved for this plate yet. Click 'Run Inference on This Plate' below.")
            viewer_status.value = "Ready"
            return
        xs = [r["x"] for r in records]
        ys = [r["y"] for r in records]
        names = [r["name"] for r in records]

    with viewer_output:
        clear_output(wait=True)

        fig, axes = plt.subplots(1, 2, figsize=(16, 7))

        axes[0].imshow(data, origin="lower", cmap="gray")
        axes[0].set_title("Raw Plate")

        axes[1].imshow(data, origin="lower", cmap="gray")
        axes[1].set_title(f"{viewer_mode.value} ({len(xs)} sources)")

        axes[1].scatter(xs, ys, s=50, facecolors="none", edgecolors="red")

        for x, y, name in zip(xs, ys, names):
            if name and name not in ("Unknown", "Unlabeled"):
                axes[1].text(
                    x + 5, y + 5, name,
                    color="yellow", fontsize=6,
                    bbox=dict(facecolor="black", alpha=0.5, edgecolor="none", pad=1)
                )

        if viewer_error_toggle.value == "Show Errors" and meta.get("errors"):
            annotate_errors(axes[1], meta["errors"])

        plt.tight_layout()

        buf = BytesIO()
        fig.savefig(buf, format="png", bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)
        display(widgets.Image(value=buf.read(), format="png"))

    viewer_status.value = f"Done | quality: {meta.get('quality', 'fair')}"

def _on_run_inference_click(_):
    fits_path = viewer_dropdown.value
    if fits_path is None:
        viewer_status.value = "No plate selected."
        return
    viewer_status.value = "Running inference..."
    xs, ys, records = run_inference_on_plate(fits_path)
    n_named = sum(1 for r in records if r["name"] != "Unknown") if records else 0
    viewer_status.value = f"Inference done: {len(xs)} star(s) found, {n_named} named."
    if viewer_mode.value == "Model Predictions":
        show_viewer_plate()

viewer_dropdown.observe(show_viewer_plate, names="value")
viewer_mode.observe(show_viewer_plate, names="value")
viewer_error_toggle.observe(show_viewer_plate, names="value")
run_inference_btn.on_click(_on_run_inference_click)

rebuild_viewer_dropdown()

display(widgets.VBox([
    widgets.HTML("<b>Plate viewer</b> (same filter/search/error overlay style as 02_A)"),
    viewer_mode,
    viewer_error_toggle,
    viewer_category_filter,
    viewer_search_box,
    viewer_dropdown,
    viewer_dropdown_status,
    viewer_status,
    run_inference_btn,
    viewer_output,
]))

if viewer_dropdown.options:
    viewer_dropdown.value = viewer_dropdown.options[0][1]